# Luxury Appliance Market Demand Analysis
## Interactive Exploration Notebook

This notebook provides an interactive environment to explore the luxury appliance market data and regression results.

**Target Brands**: Viking, Wolf, Sub-Zero, Thermador  
**Analysis Period**: 1990-Present  
**Data Source**: Federal Reserve Economic Data (FRED)

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.api import OLS, add_constant
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries loaded successfully")

## Load Data

In [ ]:
# Load quarterly data (recommended for regression)
df_quarterly = pd.read_csv('data/quarterly_data.csv', index_col=0, parse_dates=True)

# Load monthly data (for detailed time series)
df_monthly = pd.read_csv('data/monthly_data.csv', index_col=0, parse_dates=True)

print(f"Quarterly Data: {df_quarterly.shape}")
print(f"Date Range: {df_quarterly.index.min()} to {df_quarterly.index.max()}")
print(f"\nMonthly Data: {df_monthly.shape}")
print(f"Date Range: {df_monthly.index.min()} to {df_monthly.index.max()}")

## Data Overview

In [ ]:
# Display first few rows
df_quarterly.head()

In [ ]:
# Summary statistics
df_quarterly.describe()

In [ ]:
# Check for missing data
missing = pd.DataFrame({
    'Missing': df_quarterly.isna().sum(),
    'Percent': (df_quarterly.isna().sum() / len(df_quarterly) * 100).round(2)
})
missing[missing['Missing'] > 0].sort_values('Missing', ascending=False)

## Exploratory Data Analysis

In [ ]:
# Plot key drivers over time
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Key Economic Drivers (Quarterly)', fontsize=16, fontweight='bold')

# Income
axes[0, 0].plot(df_quarterly.index, df_quarterly['disposable_income'], linewidth=2)
axes[0, 0].set_title('Disposable Income')
axes[0, 0].set_ylabel('Billions of Dollars')

# Stock Market
axes[0, 1].plot(df_quarterly.index, df_quarterly['sp500'], linewidth=2, color='orange')
axes[0, 1].set_title('S&P 500')
axes[0, 1].set_ylabel('Index Value')

# Home Prices
axes[1, 0].plot(df_quarterly.index, df_quarterly['home_price_index'], linewidth=2, color='green')
axes[1, 0].set_title('Home Price Index')
axes[1, 0].set_ylabel('Index Value')

# Housing Starts
axes[1, 1].plot(df_quarterly.index, df_quarterly['housing_starts_single'], linewidth=2, color='red')
axes[1, 1].set_title('Housing Starts (Single-Family)')
axes[1, 1].set_ylabel('Thousands')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis
key_vars = [
    'pce_durables',
    'disposable_income',
    'sp500',
    'home_price_index',
    'housing_starts_single',
    'consumer_sentiment',
    'mortgage_30y'
]

corr = df_quarterly[key_vars].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True)
plt.title('Correlation Matrix: Key Drivers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Regression Analysis

In [ ]:
# Simple regression: Income & Wealth
vars_needed = ['pce_durables', 'disposable_income', 'sp500', 'home_price_index']
df_model = df_quarterly[vars_needed].dropna()

y = df_model['pce_durables']
X = df_model[['disposable_income', 'sp500', 'home_price_index']]
X = add_constant(X)

model = OLS(y, X).fit()
print(model.summary())

In [ ]:
# Plot actual vs predicted
df_model['predicted'] = model.predict(X)

plt.figure(figsize=(12, 6))
plt.plot(df_model.index, df_model['pce_durables'], label='Actual', linewidth=2)
plt.plot(df_model.index, df_model['predicted'], label='Predicted', linewidth=2, linestyle='--')
plt.title('Model Fit: Actual vs Predicted Durable Goods Spending', fontsize=14, fontweight='bold')
plt.ylabel('Billions of Dollars')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"R-squared: {model.rsquared:.4f}")
print(f"Adjusted R-squared: {model.rsquared_adj:.4f}")

## Elasticity Analysis

In [ ]:
# Log-log model for elasticities
vars_needed = ['pce_durables', 'disposable_income', 'sp500', 'home_price_index']
df_log = np.log(df_quarterly[vars_needed].dropna())

y = df_log['pce_durables']
X = df_log[['disposable_income', 'sp500', 'home_price_index']]
X = add_constant(X)

model_log = OLS(y, X).fit()
print(model_log.summary())

print("\n" + "="*60)
print("ELASTICITY INTERPRETATION")
print("="*60)
elasticities = model_log.params.drop('const')
for var, elasticity in elasticities.items():
    print(f"\n{var}:")
    print(f"  Elasticity: {elasticity:.3f}")
    if abs(elasticity) > 1:
        print(f"  → Elastic: 1% increase → {elasticity:.2f}% change in spending")
    else:
        print(f"  → Inelastic: 1% increase → {elasticity:.2f}% change in spending")

## Time Series Decomposition

In [ ]:
# Trend analysis
df_monthly['pce_durables_ma12'] = df_monthly['pce_durables'].rolling(window=12).mean()

plt.figure(figsize=(14, 6))
plt.plot(df_monthly.index, df_monthly['pce_durables'], alpha=0.5, label='Monthly')
plt.plot(df_monthly.index, df_monthly['pce_durables_ma12'], linewidth=2, label='12-Month Moving Average')
plt.title('Durable Goods Spending: Monthly vs Trend', fontsize=14, fontweight='bold')
plt.ylabel('Billions of Dollars')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Custom Analysis

Use the cells below to perform your own custom analysis:

In [ ]:
# Your custom analysis here


## Load Regression Results

In [ ]:
# Load model comparison
try:
    model_comparison = pd.read_csv('results/model_comparison.csv')
    print("Model Comparison:")
    print(model_comparison)
    
    # Plot comparison
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(model_comparison))
    width = 0.35
    
    ax.bar(x - width/2, model_comparison['R²'], width, label='R²')
    ax.bar(x + width/2, model_comparison['Adj. R²'], width, label='Adj. R²')
    
    ax.set_xlabel('Model')
    ax.set_ylabel('R² Value')
    ax.set_title('Model Comparison')
    ax.set_xticks(x)
    ax.set_xticklabels(model_comparison['Model'], rotation=45, ha='right')
    ax.legend()
    
    plt.tight_layout()
    plt.show()
    
except FileNotFoundError:
    print("Results not found. Please run: python src/run_analysis.py")

In [ ]:
# Load elasticities
try:
    elasticities = pd.read_csv('results/elasticities.csv', index_col=0)
    print("\nIncome Elasticities:")
    print(elasticities)
    
    # Plot elasticities
    elasticities.sort_values('Elasticity').plot(kind='barh', figsize=(10, 6), legend=False)
    plt.xlabel('Elasticity')
    plt.title('Income Elasticity of Luxury Appliance Spending')
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    plt.axvline(x=1, color='gray', linestyle='--', linewidth=0.5)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
except FileNotFoundError:
    print("Results not found. Please run: python src/run_analysis.py")

## Conclusions

Based on the analysis:

1. **Key Drivers**: [To be filled after running analysis]
2. **Income Elasticity**: [Value]
3. **Best Model**: [Model name and R²]
4. **Recommendations**: [Strategic insights]
